In [ ]:
# ! pip install -q transformers evaluate accelerate pillow torchvision
# ! pip install -qU datasets

## Load dependencies

In [ ]:
from functools import partial
import evaluate, torch
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor
from transformers import (
    AutoModelForImageClassification,
    pipeline,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer, 
    DefaultDataCollator, 
    AutoImageProcessor
)

## Config

In [ ]:
user_name = "amin-oj"
dataset_dict = {"path": "ethz/food101", "split":"train[:5000]"}
checkpoint = "google/vit-base-patch16-224-in21k"
eval_metric = "accuracy"
push_to_hub=True
num_train_epochs=3
train_bs = 16
eval_bs = 16
task = "image-classification"
ckpt_name = f"{checkpoint.split('/')[-1]}-SFT-{task}-{dataset_dict['path'].replace("/", "_")}"
print(ckpt_name)

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
food = load_dataset(**dataset_dict)
food = food.train_test_split(test_size=0.2)
print(food["train"][0])
labels = food["train"].features["label"].names
print(f"10 example labels: {labels[:10]}")

label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

## Preprocess

In [ ]:
# load the model's processor
image_processor = AutoImageProcessor.from_pretrained(checkpoint, use_fast=True)
print(f"image_processor.image_mean: {image_processor.image_mean}")
print(f"image_processor.image_std: {image_processor.image_std}")
print(f"image_processor.size: {image_processor.size}")

In [ ]:
# Apply some image transformations to the images to make the model more robust against overfitting.
# using torchvision's transforms
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])


def transforms(examples):
    examples["pixel_values"] = [
        _transforms(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

# apply the preprocessing function over the entire dataset
# Unlike the .map() method, which applies transformations eagerly and caches the results to disk
# The with_transform() method applies the transformation on-the-fly when a batch is accessed (lazy)
food = food.with_transform(transforms)

## Train

In [ ]:
data_collator = DefaultDataCollator()
# DefaultDataCollator does not apply additional preprocessing such as padding.

def compute_metrics(eval_pred, eval_metric):
    accuracy = evaluate.load(eval_metric)
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    num_train_epochs=num_train_epochs,
    metric_for_best_model=eval_metric,
    push_to_hub=push_to_hub,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    remove_unused_columns=False,    # otherwise, "image" is dropped and
                                    # We won't have "pixel_values", which are calculated lazily.
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=partial(compute_metrics, eval_metric=eval_metric),
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Inference

In [ ]:
ds = load_dataset("ethz/food101", split="validation[:10]")
image = ds["image"][0]
my_ckpt = f"{user_name}/{ckpt_name}"

In [ ]:
# import matplotlib.pyplot as plt
# plt.imshow(image)

### The simplest way

In [ ]:
classifier = pipeline(task, model=my_ckpt)
classifier(image)

### More complex | More flexible

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(my_ckpt)
inputs = image_processor(image, return_tensors="pt")
model = AutoModelForImageClassification.from_pretrained(my_ckpt)
with torch.no_grad():
    logits = model(**inputs).logits
predicted_label = logits.argmax(-1).item()
model.config.id2label[predicted_label]